# Task 4: Advanced Technique — MLflow Experiment Tracking

**Why this technique was selected:** the task brief explicitly recommends MLflow because it connects directly to the MLOps component of the programme, and — unlike the other options (GAN, quantisation, self-supervised anomaly detection) — it doesn't require building an entirely new model. Instead it wraps the models already built in Task 1 (Linear/Logistic Regression, Random Forest Regressor/Classifier) with systematic experiment tracking: every run's parameters, metrics, and the trained model itself are logged automatically, rather than being copy-pasted by hand between notebook cells and a report.

**How it was implemented:** each model from Task 1 is re-trained inside an `mlflow.start_run()` context. Model hyperparameters are logged with `mlflow.log_param`, evaluation metrics with `mlflow.log_metric`, and the fitted model itself as a versioned artifact with `mlflow.sklearn.log_model`. Runs are grouped under a single named experiment (`metro_traffic_models`) so all four models — regression and classification — can be compared side by side afterwards, either through `mlflow.search_runs()` (used below to pull results back into this notebook) or the `mlflow ui` web dashboard.

**What value it adds:** it replaces manual, error-prone copy-pasting of metrics between runs with a queryable, versioned log — every run's exact hyperparameters, metrics, and the resulting model artifact are all permanently recorded together, so results are fully reproducible and easy to compare later (e.g. "did Random Forest with max_depth=12 actually beat max_depth=8?"). It's also the natural first building block toward real MLOps: the logged model artifacts are already in a format (`mlflow.sklearn`) that can be registered and served without re-writing any code.

**Limitations:** MLflow only tracks and organises experiments — it does not improve model performance, select features, or tune hyperparameters on its own. This notebook uses the default local file-based tracking store (`./mlruns/`), which is fine for a single-person capstone project but doesn't support real team collaboration or remote access the way a hosted MLflow tracking server would. It also doesn't version the underlying dataset itself (a separate concern, e.g. DVC), so "reproducibility" here covers the model and its metrics, not guaranteeing the exact same `Features_Metro_Interstate_Traffic_Volume.csv` was used run to run unless that's tracked separately. Finally, logging discipline is manual — nothing forces every future experiment to actually go through MLflow rather than being run ad hoc outside it.

## 1. Imports

In [1]:
# pip install mlflow   (not in the standard scientific stack)
import logging

import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

RANDOM_STATE = 42

## 2. Logging setup
A named logger (logging.getLogger("task1_models")) is used instead of the root logger, with propagate = False, so this doesn't collide with Jupyter's own logging setup or double-print.

In [2]:
logger = logging.getLogger("task4_mlflow")


def setup_logging():
    if logger.handlers:
        return
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    fh = logging.FileHandler("pipeline.log", mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    logger.propagate = False


setup_logging()
logger.info("Logging initialised")

2026-09-23 23:22:57 | INFO | task4_mlflow | Logging initialised


## 3. Load data and rebuild the Task 1 feature set / targets

In [5]:
FEATURES_PATH = "Features_Metro_Interstate_Traffic_Volume.csv"

df = pd.read_csv(FEATURES_PATH, parse_dates=["date_time"])
logger.info("Loaded %s rows, %s columns from %s", len(df), df.shape[1], FEATURES_PATH)

# --- is_holiday flag ---
is_holiday = df["holiday"].notna() & (df["holiday"] != "None")
df["is_holiday"] = is_holiday.astype(int)

# --- accident_risk proxy label (congestion + adverse weather), same as Task 1 ---
congestion_cutoff = df["traffic_volume"].quantile(2 / 3)
is_congested = df["traffic_volume"] >= congestion_cutoff
is_adverse_weather = (
    (df["is_severe_weather"] == 1) | (df["is_precipitation"] == 1) | (df["is_overcast"] == 1)
)
df["accident_risk"] = (is_congested & is_adverse_weather).astype(int)

# --- Shared feature set (traffic_volume and anything derived from it excluded) ---
time_features = [
    "hour", "day_of_week", "is_weekend",
    "hour_sin", "hour_cos"
]
exclude_raw_text = {"weather_main", "weather_description"}
weather_onehot = sorted(
    c for c in df.columns if c.startswith("weather_") and c not in exclude_raw_text
)
weather_derived = [
    "is_precipitation", "total_precipitation", "is_overcast",
    "is_clear", "is_severe_weather", "clouds_all", "temp_c",
]
feature_cols = time_features + weather_onehot + weather_derived + ["is_holiday"]

model_df = df[feature_cols + ["traffic_volume", "accident_risk"]].dropna()
logger.info("Modelling on %s rows, %s features", len(model_df), len(feature_cols))

2026-09-23 23:23:34 | INFO | task4_mlflow | Loaded 48187 rows, 38 columns from Features_Metro_Interstate_Traffic_Volume.csv
2026-09-23 23:23:34 | INFO | task4_mlflow | Modelling on 48187 rows, 24 features


## 4. Configure MLflow

Uses the default local file store (`./mlruns/`) — no separate tracking server needed for this project. All runs below are grouped under one named experiment so they can be compared together afterwards.

**Why an absolute URI, not `"file:./mlruns"`:** a *relative* `file:` URI is a well-known source of `MlflowException` errors raised deep inside `_resolve_tracking_uri` / `_get_store_with_resolved_uri` — MLflow has to turn the string into a filesystem path itself, and that resolution is sensitive to exactly what the notebook process's working directory is when the kernel started (this bites Windows users especially, since a Windows path doesn't turn into a valid `file:` URI just by prefixing it). Building the URI explicitly from an **absolute**, resolved path with `Path.as_uri()` sidesteps that class of failure entirely, and creating the folder first (`mkdir(exist_ok=True)`) avoids the other common cause of this same exception: MLflow trying to open a store folder that only partially exists (e.g. left over from an interrupted earlier run). If you still hit an `MlflowException` here after this change, the fix is almost always to delete the `mlruns/` folder printed below and re-run this cell — a corrupted/partial store is the most common remaining cause.

In [7]:
import pathlib
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlruns_dir = pathlib.Path("mlruns").resolve()
mlruns_dir.mkdir(exist_ok=True)

mlflow.set_tracking_uri(mlruns_dir.as_uri())
mlflow.set_experiment("metro_traffic_models")
logger.info("MLflow experiment set: metro_traffic_models (tracking URI: %s)", mlruns_dir.as_uri())
print(f"MLflow tracking store: {mlruns_dir}")

2026/09/23 23:31:24 INFO mlflow.tracking.fluent: Experiment with name 'metro_traffic_models' does not exist. Creating a new experiment.
2026-09-23 23:31:24 | INFO | task4_mlflow | MLflow experiment set: metro_traffic_models (tracking URI: file:///C:/Projects/Learning/NUS_AI_ML_Data_Science_Programme/33rd_week_capstone_project/smart_traffic_capstone_project/smart_traffic_capstone_project/part3_machine_learning/notebooks/mlruns)


MLflow tracking store: C:\Projects\Learning\NUS_AI_ML_Data_Science_Programme\33rd_week_capstone_project\smart_traffic_capstone_project\smart_traffic_capstone_project\part3_machine_learning\notebooks\mlruns


## 5. Regression runs (target: `traffic_volume`)

Same train/test split and models as Task 1 — now each one is wrapped in an `mlflow.start_run()` block so its parameters, metrics, and the model artifact itself are all logged together.

In [8]:
X = model_df[feature_cols]
y_reg = model_df["traffic_volume"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

regressors = {
    "Linear Regression": (LinearRegression(), {}),
    "Random Forest Regressor": (
        RandomForestRegressor(
            n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
        ),
        {"n_estimators": 200, "max_depth": 12},
    ),
}

for name, (model, extra_params) in regressors.items():
    with mlflow.start_run(run_name=f"regression_{name}"):
        mlflow.set_tag("task", "regression")
        mlflow.log_param("model_type", name)
        for k, v in extra_params.items():
            mlflow.log_param(k, v)

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("R2", r2)
        mlflow.sklearn.log_model(model, "model")

        logger.info("[MLflow] regression/%s — MAE: %.2f, R2: %.4f", name, mae, r2)

2026/09/23 23:31:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-23 23:32:11 | INFO | task4_mlflow | [MLflow] regression/Linear Regression — MAE: 820.40, R2: 0.7219
2026/09/23 23:32:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-23 23:32:32 | INFO | task4_mlflow | [MLflow] regression/Random Forest Regressor — MAE: 263.71, R2: 0.9480


## 6. Classification runs (target: `accident_risk` proxy label)

In [9]:
y_clf = model_df["accident_risk"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

classifiers = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, class_weight="balanced"),
        {"max_iter": 1000, "class_weight": "balanced"},
    ),
    "Random Forest Classifier": (
        RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        {"n_estimators": 200, "max_depth": 12, "class_weight": "balanced"},
    ),
}

for name, (model, extra_params) in classifiers.items():
    with mlflow.start_run(run_name=f"classification_{name}"):
        mlflow.set_tag("task", "classification")
        mlflow.log_param("model_type", name)
        for k, v in extra_params.items():
            mlflow.log_param(k, v)

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        auc = roc_auc_score(y_test, probs)

        mlflow.log_metric("Accuracy", acc)
        mlflow.log_metric("Precision", prec)
        mlflow.log_metric("Recall", rec)
        mlflow.log_metric("F1", f1)
        mlflow.log_metric("ROC_AUC", auc)
        mlflow.sklearn.log_model(model, "model")

        logger.info(
            "[MLflow] classification/%s — Acc: %.4f, Prec: %.4f, Rec: %.4f, F1: %.4f, AUC: %.4f",
            name, acc, prec, rec, f1, auc,
        )

2026/09/23 23:32:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-23 23:32:51 | INFO | task4_mlflow | [MLflow] classification/Logistic Regression — Acc: 0.9104, Prec: 0.6095, Rec: 0.9544, F1: 0.7439, AUC: 0.9597
2026/09/23 23:32:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-09-23 23:33:16 | INFO | task4_mlflow | [MLflow] classification/Random Forest Classifier — Acc: 0.9511, Prec: 0.7414, Rec: 0.9856, F1: 0.8462, AUC: 0.9924


## 7. Compare logged runs

Pulls every run back from the MLflow tracking store into a DataFrame — no need to leave the notebook or open the MLflow UI just to compare results (though `mlflow ui` run from a terminal in this folder gives a fuller interactive dashboard, useful for a report screenshot).

In [10]:
runs = mlflow.search_runs(experiment_names=["metro_traffic_models"])
cols = [c for c in runs.columns if c.startswith("tags.task") or c.startswith("params.") or c.startswith("metrics.") or c == "run_id"]
display_cols = ["tags.task", "params.model_type"] + [c for c in cols if c.startswith("metrics.")]
runs[display_cols].sort_values(["tags.task", "params.model_type"])

,tags.task,params.model_type,metrics.Recall,metrics.F1,metrics.ROC_AUC,metrics.Precision,metrics.Accuracy,metrics.R2,metrics.MAE
1,classification,Logistic Regression,0.954373,0.743924,0.959655,0.609519,0.910355,NaN,NaN
0,classification,Random Forest Classifier,0.985551,0.846229,0.992362,0.741419,0.951131,NaN,NaN
3,regression,Linear Regression,NaN,NaN,NaN,NaN,NaN,0.721930,820.397637
2,regression,Random Forest Regressor,NaN,NaN,NaN,NaN,NaN,0.947978,263.706850


## 8. Viewing the full dashboard

From a terminal, in the same folder as this notebook, run:

```
mlflow ui
```

then open `http://localhost:5000` — this shows every run with sortable/filterable columns for every logged parameter and metric, plots comparing runs, and the downloadable model artifacts. A screenshot of this dashboard, alongside the comparison table above, is good evidence for the report of the experiment-tracking requirement being met.